# Saving Scraped Real Estate Data into a Database

In this notebook, we take the cleaned real estate listings data scraped from Bayut
and store it in a SQLite database.

Why this step is important:
- CSV files are good for storage, but databases are better for querying and analysis
- Databases allow dashboards, analytics, and ML pipelines to scale
- This creates a persistent data layer for the rest of the project


## Imports & setup

In [10]:
import pandas as pd
import sqlite3  # DB we're using
import os

## Load Cleaned Data

We load the cleaned CSV file generated in the scraping stage.
This dataset has already been:
- Cleaned from duplicates
- Normalized
- Converted into numeric formats

In [11]:
# Define data folder and CSV path (relative)
data_folder = "data"
csv_file = "jvc_apartments_cleaned.csv"
csv_path = os.path.join("..", data_folder, csv_file)

# Check if file exists
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"{csv_path} not found. Make sure the CSV is in the data folder.")

# Load CSV
df = pd.read_csv(csv_path)

# Quick sanity check
print(f"Dataset loaded: {csv_path}")
print("Dataset shape:", df.shape)
df.head()

Dataset loaded: ..\data\jvc_apartments_cleaned.csv
Dataset shape: (484, 13)


,title,price,frequency,bedrooms,bathrooms,area,location,url,price_clean,price_yearly_aed,bedrooms_clean,bathrooms_clean,area_clean
0,Refined 2-Bed Apartment | Premium Finishes | E...,"89,999",yearly,2,3,"1,128 sqft","Rose 10, JVC District 11, Jumeirah Village Cir...",https://www.bayut.com/property/details-1358095...,89999,89999,2.0,3.0,1128.0
1,Unique Layout I Brand New Building I Closed Ki...,"119,990",yearly,2,3,"1,300 sqft","SH Living 1, JVC District 14, Jumeirah Village...",https://www.bayut.com/property/details-1296493...,119990,119990,2.0,3.0,1300.0
2,Pool Access - Spacious layout - With Appliances,"90,000",yearly,1,2,"1,056 sqft","Westview Garden, JVC District 15, Jumeirah Vil...",https://www.bayut.com/property/details-1358940...,90000,90000,1.0,2.0,1056.0
3,Book Now -Pool View-1/2/3 BHK,"88,000",yearly,1,2,890 sqft,"Westview Garden, JVC District 15, Jumeirah Vil...",https://www.bayut.com/property/details-1353901...,88000,88000,1.0,2.0,890.0
4,Chiller W Dewa-Spacious Layout-Multiple Options,"130,000",yearly,2,3,"1,470 sqft","Westview Garden, JVC District 15, Jumeirah Vil...",https://www.bayut.com/property/details-1353901...,130000,130000,2.0,3.0,1470.0


In [12]:
import os
print(os.getcwd())


c:\Users\temps\Desktop\M2\Web Scraping\Web Scraping Project\RealEstate_Analytics\notebooks


## Create SQLite Database

SQLite is a serverless relational database.
The database will be saved as a `.db` file inside the data folder.


In [13]:
# Database path
db_path = os.path.join("..", "data", "database.db")

# Create connection
conn = sqlite3.connect(db_path)

# Save DataFrame to SQL table
df.to_sql(
    name="jvc_apartments",
    con=conn,
    if_exists="replace",
    index=False
)

print("Data successfully written to SQLite database.")


Data successfully written to SQLite database.


## Verifying Database Contents

_We run a simple SQL query to confirm that the data was inserted correctly._

In [14]:
query = "SELECT COUNT(*) AS total_rows FROM jvc_apartments"
row_count = pd.read_sql(query, conn)

row_count


,total_rows
0,484


_Sample query, just to check if the writing was successful_

In [15]:
sample_query = """
SELECT
    bedrooms_clean,
    AVG(price_yearly_aed) AS avg_yearly_price
FROM jvc_apartments
GROUP BY bedrooms_clean
ORDER BY bedrooms_clean
"""

pd.read_sql(sample_query, conn)


,bedrooms_clean,avg_yearly_price
0,1.0,83534.270893
1,2.0,113542.194690
2,3.0,197307.076923
3,4.0,206999.900000
4,5.0,320000.000000


_This query shows databases can be used for analytical queries, like understanding how prices vary by number of bedrooms_

## Close Connection

Closing DB connection is best practice to avoid file locks or corruption

In [16]:
conn.close()
print("Database connection closed.")


Database connection closed.


## Summary

In this notebook:
- Loaded cleaned real estate listings data
- Created a SQLite database
- Stored the data in a relational table
- Verified successful insertion using SQL queries

This database will be used in the next stage for:
- Interactive dashboards
- Data preprocessing
- Machine learning models
